Stay Empty


In [ ]:
import requests
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()

#Search for project folder
project_root = next(
    (
        folder
        for folder in [current_dir, *current_dir.parents]
        if (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Projektordner nicht gefunden")

# folder for raw data
raw_dir = project_root / "data" / "raw"

raw_dir.mkdir(parents=True, exist_ok=True)

print("Projektordner:", project_root)
print("Rohdatenordner:", raw_dir)


Projektordner: c:\Projects\velo-wetter-zuerich-HS26
Rohdatenordner: c:\Projects\velo-wetter-zuerich-HS26\data\raw


In [5]:
# CKAN API of Stadt Zürich
api_url = (
    "https://ckan-prod.zurich.datopian.com"
    "/api/3/action/package_show"
)

# Unqiue identifier of the dataset
dataset_id = (
    "ted_taz_verkehrszaehlungen_werte_fussgaenger_velo"
)

# send request to CKAN API to get dataset information
response = requests.get(
    api_url,
    params={"id": dataset_id},
    timeout=30
)

# raise an exception if the request was not successful
response.raise_for_status()

# parse the response as JSON
data = response.json()

if not data.get("success"):
    raise RuntimeError("API doesnt return success")

# extract the dataset information from the response
dataset = data["result"]

print("API Connection successful")
print("Dataset:", dataset["title"])

API Connection successful
Dataset: Daten der automatischen Fussgänger- und Velozählung - Viertelstundenwerte


In [6]:
# Alle Datenressourcen abrufen
resources = dataset["resources"]

# Unsere gewünschten Jahre
years = [2023, 2024, 2025]

# Downloadinformationen sammeln
download_resources = []

for year in years:

    matching = [
        resource
        for resource in resources
        if resource.get("name", "").startswith(f"{year}_")
        and resource.get("format", "").lower() == "csv"
    ]

    if len(matching) != 1:
        raise ValueError(
            f"Für {year} wurden {len(matching)} "
            "passende CSV-Dateien gefunden."
        )

    resource = matching[0]

    download_resources.append({
        "year": year,
        "resource_id": resource["id"],
        "url": resource["url"]
    })

# Ergebnisse anzeigen
display(pd.DataFrame(download_resources))

,year,resource_id,url
0,2023,8bb6b21f-e412-4710-a50f-26202b2d1ca3,https://data.stadt-zuerich.ch/dataset/ted_taz_...
1,2024,1e1865cc-abc1-4dae-b761-030f7aaff787,https://data.stadt-zuerich.ch/dataset/ted_taz_...
2,2025,9e535ea2-7273-4474-94f7-ac64d2f16ae6,https://data.stadt-zuerich.ch/dataset/ted_taz_...


In [7]:
for resource in download_resources:

    year = resource["year"]
    url = resource["url"]

    file_path = raw_dir / f"velo_{year}.csv"

    # Vorhandene Dateien nicht überschreiben
    if file_path.exists():
        print(f"{year}: Datei existiert bereits.")
        continue

    print(f"Lade Velodaten {year} herunter ...")

    # Temporäre Datei verwenden
    temp_path = raw_dir / f"velo_{year}.csv.part"

    with requests.get(
        url,
        stream=True,
        timeout=(30, 180)
    ) as response:

        response.raise_for_status()

        with temp_path.open("wb") as file:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)

    # Nach erfolgreichem Download umbenennen
    temp_path.replace(file_path)

    print(f"{year}: Download erfolgreich!")

print("Downloadvorgang abgeschlossen.")

Lade Velodaten 2023 herunter ...
2023: Download erfolgreich!
Lade Velodaten 2024 herunter ...
2024: Download erfolgreich!
Lade Velodaten 2025 herunter ...
2025: Download erfolgreich!
Downloadvorgang abgeschlossen.


In [8]:
for year in years:

    file_path = raw_dir / f"velo_{year}.csv"

    if file_path.exists():

        file_size = (
            file_path.stat().st_size / (1024 * 1024)
        )

        print(
            f"{year}: Vorhanden, "
            f"{file_size:.2f} MB"
        )

    else:
        print(f"{year}: Datei fehlt!")

2023: Vorhanden, 41.12 MB
2024: Vorhanden, 39.78 MB
2025: Vorhanden, 35.88 MB


In [9]:
# Ressourcen-ID für das Jahr 2025
resource_id = next(
    r["resource_id"]
    for r in download_resources
    if r["year"] == 2025
)

# Daten-API
data_api_url = (
    "https://ckan-prod.zurich.datopian.com"
    "/api/action/datastore_search"
)

# Fünf Messdatensätze abrufen
response = requests.get(
    data_api_url,
    params={
        "resource_id": resource_id,
        "limit": 5
    },
    timeout=30
)

response.raise_for_status()

api_result = response.json()

if not api_result.get("success"):
    raise RuntimeError("Daten-API meldet einen Fehler")

records = api_result["result"]["records"]

display(pd.DataFrame(records))

,_id,FK_STANDORT,DATUM,VELO_IN,VELO_OUT,FUSS_IN,FUSS_OUT,OST,NORD
0,1,4241,2025-01-01T00:00:00,3,0,None,None,2682297,1248328
1,2,2989,2025-01-01T00:00:00,1,2,None,None,2682278,1248324
2,3,2991,2025-01-01T00:00:00,18,2,None,None,2682756,1247323
3,4,4255,2025-01-01T00:00:00,0,0,None,None,2682881,1246549
4,5,4242,2025-01-01T00:00:00,1,0,None,None,2682337,1248451
